# Module 7 — Deep Neural Networks for Regression and Classification

We build deep neural networks on the Boston Housing dataset to tackle two problems:
- **Part 1 (Regression):** Predict the median home value (MEDV)
- **Part 2 (Classification):** Predict whether a home is *expensive* (MEDV above median)

For each part we train a baseline model, then a tuned model, and compare performance.

## Step 1 — Install Packages

In [ ]:
import subprocess, sys

packages = ["tensorflow", "numpy", "pandas", "matplotlib", "seaborn", "scikit-learn"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg],
                          stderr=subprocess.DEVNULL)
print("All packages ready.")

## Step 2 — Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_squared_error, r2_score,
                              accuracy_score, classification_report, confusion_matrix)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)
print(f"TensorFlow {tf.__version__}")

## Step 3 — Load the Boston Housing Dataset

We load from `tf.keras.datasets.boston_housing`. If unavailable in newer TF, we fall back to a CSV mirror.

In [ ]:
try:
    (X_raw, y_raw), _ = tf.keras.datasets.boston_housing.load_data(seed=42)
    feature_names = ["CRIM","ZN","INDUS","CHAS","NOX","RM","AGE","DIS","RAD","TAX","PTRATIO","B","LSTAT"]
    print("Loaded from tf.keras.datasets.boston_housing")
except Exception:
    import urllib.request
    url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
    urllib.request.urlretrieve(url, "boston.csv")
    df_raw = pd.read_csv("boston.csv")
    feature_names = df_raw.columns[:-1].tolist()
    X_raw = df_raw.iloc[:, :-1].values
    y_raw = df_raw.iloc[:, -1].values
    print("Loaded from CSV fallback")

df = pd.DataFrame(X_raw, columns=feature_names)
df["MEDV"] = y_raw
print(f"Shape: {df.shape}")
df.head()

## Step 4 — Exploratory Data Analysis

In [ ]:
print(df.describe().round(2))

In [ ]:
print("Missing values:", df.isnull().sum().sum())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["MEDV"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Distribution of MEDV (House Price $K)")
axes[0].set_xlabel("MEDV"); axes[0].set_ylabel("Count")

corr = df.corr()["MEDV"].drop("MEDV").sort_values()
corr.plot(kind="barh", ax=axes[1], color="steelblue")
axes[1].set_title("Feature Correlation with MEDV")
axes[1].axvline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.show()

**Key observations:**
- `LSTAT` (% lower-status population) and `RM` (avg rooms) are the strongest predictors.
- No missing values — no imputation needed.

---
# Part 1: Regression — Predicting Median House Value

## Step 5 — Preprocess for Regression

Normalize features with `StandardScaler` and split 80/20.

In [ ]:
X = df[feature_names].values
y_reg = df["MEDV"].values

X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape} | Test: {X_test_s.shape}")

## Step 6 — Baseline Regression Model

A simple 2-hidden-layer network (64 → 32 → 1) with ReLU activations and a linear output.

In [ ]:
def build_baseline_reg(input_dim):
    model = keras.Sequential([
        layers.Dense(64, activation="relu", input_shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(1)  # linear output for regression
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

baseline_reg = build_baseline_reg(X_train_s.shape[1])
baseline_reg.summary()

## Step 7 — Train & Evaluate Baseline Regression

In [ ]:
history_base_reg = baseline_reg.fit(
    X_train_s, y_train,
    epochs=100, batch_size=32,
    validation_split=0.15,
    verbose=0
)

y_pred_base = baseline_reg.predict(X_test_s).flatten()
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
r2_base   = r2_score(y_test, y_pred_base)
print(f"Baseline Regression → RMSE: {rmse_base:.2f} | R²: {r2_base:.4f}")

## Step 8 — Tuned Regression Model

Improvements: wider/deeper network (128 → 64 → 32 → 1), `BatchNormalization`, `Dropout(0.2)`, lower learning rate, and `EarlyStopping`.

In [ ]:
def build_tuned_reg(input_dim):
    model = keras.Sequential([
        layers.Dense(128, activation="relu", input_shape=(input_dim,)),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse", metrics=["mae"]
    )
    return model

tuned_reg = build_tuned_reg(X_train_s.shape[1])
tuned_reg.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)

history_tuned_reg = tuned_reg.fit(
    X_train_s, y_train,
    epochs=300, batch_size=32,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=0
)

y_pred_tuned = tuned_reg.predict(X_test_s).flatten()
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
r2_tuned   = r2_score(y_test, y_pred_tuned)
print(f"Tuned Regression  → RMSE: {rmse_tuned:.2f} | R²: {r2_tuned:.4f}")

## Step 9 — Compare Regression Models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, h, title in zip(axes, [history_base_reg, history_tuned_reg], ["Baseline", "Tuned"]):
    ax.plot(h.history["loss"], label="Train Loss")
    ax.plot(h.history["val_loss"], label="Val Loss")
    ax.set_title(f"{title} — Loss (MSE)")
    ax.set_xlabel("Epoch"); ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, preds, title, rmse, r2 in zip(
    axes, [y_pred_base, y_pred_tuned], ["Baseline", "Tuned"],
    [rmse_base, rmse_tuned], [r2_base, r2_tuned]
):
    ax.scatter(y_test, preds, alpha=0.6, color="steelblue")
    lo, hi = y_test.min(), y_test.max()
    ax.plot([lo, hi], [lo, hi], "r--")
    ax.set_title(f"{title}  RMSE={rmse:.2f}, R²={r2:.3f}")
    ax.set_xlabel("Actual MEDV"); ax.set_ylabel("Predicted MEDV")
plt.tight_layout()
plt.show()

In [ ]:
summary_reg = pd.DataFrame({
    "Model": ["Baseline", "Tuned"],
    "RMSE":  [round(rmse_base, 3), round(rmse_tuned, 3)],
    "R2":    [round(r2_base, 3),   round(r2_tuned, 3)],
})
print("Regression Performance")
print(summary_reg.to_string(index=False))

---
# Part 2: Classification — Expensive vs Not Expensive

## Step 10 — Create Binary Target

A house is **expensive (1)** if MEDV is above the dataset median; otherwise **not expensive (0)**.

In [ ]:
threshold = np.median(y_reg)
print(f"Classification threshold (median MEDV): ${threshold:.1f}K")

y_cls = (y_reg > threshold).astype(int)
print(f"Expensive (1): {y_cls.sum()} | Not Expensive (0): {(y_cls == 0).sum()}")

# Stratified split to preserve class balance
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls)
X_tr_c = scaler.transform(X_tr_c)
X_te_c = scaler.transform(X_te_c)
print(f"Train: {X_tr_c.shape} | Test: {X_te_c.shape}")

## Step 11 — Baseline Classification Model

Same architecture as the regression baseline, with a sigmoid output.

In [ ]:
def build_baseline_cls(input_dim):
    model = keras.Sequential([
        layers.Dense(64, activation="relu", input_shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

baseline_cls = build_baseline_cls(X_tr_c.shape[1])
baseline_cls.summary()

## Step 12 — Train & Evaluate Baseline Classification

In [ ]:
history_base_cls = baseline_cls.fit(
    X_tr_c, y_tr_c,
    epochs=100, batch_size=32,
    validation_split=0.15,
    verbose=0
)

y_pred_base_cls = (baseline_cls.predict(X_te_c).flatten() > 0.5).astype(int)
acc_base_cls = accuracy_score(y_te_c, y_pred_base_cls)
print(f"Baseline Classification → Accuracy: {acc_base_cls:.4f}")
print(classification_report(y_te_c, y_pred_base_cls, target_names=["Not Expensive", "Expensive"]))

## Step 13 — Tuned Classification Model

Same tuning strategy: deeper, `BatchNormalization`, `Dropout(0.3)`, lower learning rate, `EarlyStopping`.

In [ ]:
def build_tuned_cls(input_dim):
    model = keras.Sequential([
        layers.Dense(128, activation="relu", input_shape=(input_dim,)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0005),
        loss="binary_crossentropy", metrics=["accuracy"]
    )
    return model

tuned_cls = build_tuned_cls(X_tr_c.shape[1])
tuned_cls.summary()

In [ ]:
early_stop_cls = keras.callbacks.EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True)

history_tuned_cls = tuned_cls.fit(
    X_tr_c, y_tr_c,
    epochs=300, batch_size=32,
    validation_split=0.15,
    callbacks=[early_stop_cls],
    verbose=0
)

y_pred_tuned_cls = (tuned_cls.predict(X_te_c).flatten() > 0.5).astype(int)
acc_tuned_cls = accuracy_score(y_te_c, y_pred_tuned_cls)
print(f"Tuned Classification  → Accuracy: {acc_tuned_cls:.4f}")
print(classification_report(y_te_c, y_pred_tuned_cls, target_names=["Not Expensive", "Expensive"]))

## Step 14 — Compare Classification Models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, h, title in zip(axes, [history_base_cls, history_tuned_cls], ["Baseline", "Tuned"]):
    ax.plot(h.history["accuracy"], label="Train Accuracy")
    ax.plot(h.history["val_accuracy"], label="Val Accuracy")
    ax.set_title(f"{title} — Accuracy")
    ax.set_xlabel("Epoch"); ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, preds, title, acc in zip(
    axes, [y_pred_base_cls, y_pred_tuned_cls], ["Baseline", "Tuned"],
    [acc_base_cls, acc_tuned_cls]
):
    cm = confusion_matrix(y_te_c, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Not Exp", "Expensive"],
                yticklabels=["Not Exp", "Expensive"])
    ax.set_title(f"{title} — Accuracy: {acc:.3f}")
    ax.set_ylabel("Actual"); ax.set_xlabel("Predicted")
plt.tight_layout()
plt.show()

---
# Final Summary

In [ ]:
print("=" * 55)
print("REGRESSION RESULTS (lower RMSE, higher R2 = better)")
print("=" * 55)
print(f"  Baseline : RMSE={rmse_base:.3f}  R2={r2_base:.3f}")
print(f"  Tuned    : RMSE={rmse_tuned:.3f}  R2={r2_tuned:.3f}")
print()
print("=" * 55)
print("CLASSIFICATION RESULTS (higher accuracy = better)")
print("=" * 55)
print(f"  Baseline : Accuracy={acc_base_cls:.3f}")
print(f"  Tuned    : Accuracy={acc_tuned_cls:.3f}")
print("=" * 55)

## Conclusion

**Regression:** The tuned model reduces prediction error through a deeper architecture and regularization. BatchNormalization stabilizes training; Dropout prevents overfitting on the small dataset.

**Classification:** Labeling homes as expensive/not-expensive based on the median creates a balanced task. The tuned model achieves higher accuracy with the same regularization strategy.

**Future work:**
- Systematic hyperparameter search with Keras Tuner or Optuna
- Try different activation functions (LeakyReLU, ELU, SELU)
- Ensemble methods to further reduce variance